## Welcome to this tutorial, where we will learn how to coarse-grain an all-atom structure and embed it in a membrane to run a molecular dynamics simulation.


## Tutorial setup

Run this cell **first**. It does two things:

1. Defines the GROMACS module line used everywhere, so you load it the same way in every step. Note that since this is a python environment running bash command, the **module load** command will be used everytime we want to use GROMACS.
2. Builds **short** versions of the NVT / NPT / production `.mdp` files so the whole workflow runs in a few minutes on the GPU instead of hours. These short runs are for *demonstration only* — real science needs the full-length runs in `../inp_files/`.

Run lengths chosen for the tutorial (Martini, dt = 20 fs):

| Step |     steps |   time |
| ---- |     ----- |   ---- |
| NVT  |   25 000  | 0.5 ns |
| NPT  |   50 000  |  1  ns |
| Prod | 1 000 000 |  20 ns |


In [ ]:
%%bash
# ===== Tutorial setup: create short .mdp files for a fast demo run =====
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP

INP=../inp_files

# Short NVT: 0.5 ns
sed 's/^nsteps.*/nsteps = 25000/'  $INP/CG_nvt.mdp  > CG_nvt_tut.mdp
# Short NPT: 1 ns
sed 's/^nsteps.*/nsteps = 50000/'  $INP/CG_npt.mdp  > CG_npt_tut.mdp
# Short production: 20 ns
sed 's/^nsteps.*/nsteps = 1000000/' $INP/CG_prod.mdp > CG_prod_tut.mdp
sed -i 's/^nstxout-compressed.*/nstxout-compressed       = 25000 ; 0.5 ns/' CG_prod_tut.mdp

echo 'Short mdp files created:'
for f in CG_nvt_tut CG_npt_tut CG_prod_tut; do
  printf '  %-16s ' "$f.mdp"; grep '^nsteps' $f.mdp
done

# System preparation

## Converting all atom to coarse grain

To start, we will take the all-atom structure we worked with in the previous tutorial. To convert it into a coarse-grained (CG) model, we will use a popular tool called `martinize2`. This approach is compatible with the MARTINI force field.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
martinize2 -f aa.pdb -x CG.pdb -o topol.top -dssp -p backbone -elastic

| Option         | Description                                                                                                                                                                 |
| -------------- | --------------------------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `-f aa.pdb`    | Specifies the input all-atom protein structure in PDB format.                                                                                                               |
| `-x CG.pdb`    | Writes the generated coarse-grained coordinates to `CG.pdb`.                                                                                                                |
| `-o topol.top` | Outputs the GROMACS topology file containing molecular definitions and bonded interactions.                                                                                 |
| `-dssp`        | Automatically assigns protein secondary structure using DSSP. The assigned secondary structure determines the backbone bead types and bonded parameters in the MARTINI force field. |
| `-p backbone`  | Generates position restraints for backbone beads, helping maintain the protein structure during equilibration.                                                              |
| `-elastic`     | Builds an Elastic Network Model (ENM) by adding harmonic springs between nearby backbone beads to preserve the native tertiary structure during coarse-grained simulations. |

---

Files generated are :

- `CG.pdb`: contains the coarse-grained structure file  
- `topol.top`: topology file contains information linked to all the forcefield files  
- `molecule_0.itp`: contains forcefield information for the protein  

The generated forcefield file contains:

- Bead definitions  
- Charges  
- Masses  
- Bond parameters  
- Angle parameters  
- Dihedral parameters  
- Elastic network springs (if `-elastic` is used)  
- Position restraints (if `-p backbone` is used)  

`-dssp`: The MARTINI force field assigns different backbone bead types depending on the protein's secondary structure.  

| Secondary Structure | Example Backbone Bead Type |
| ------------------- | -------------------------- |
| α-Helix             | `N0`                       |
| β-Sheet             | `Nda`                      |
| Coil / Loop         | `P5`                       |

`-elastic`: Coarse-grained models generally have smoother energy landscapes and fewer interaction sites than atomistic models, which can make proteins more flexible than expected.  

The Elastic Network Model (ENM):

- Preserves the native tertiary structure  
- Prevents excessive unfolding during simulations  
- Allows local fluctuations while maintaining the overall fold  

Elastic springs are typically created between backbone beads within a predefined distance range.  

---

## Modifying topology files to suit best for our simulations

We need to modify `topol.top` as the file generated is a template and missing some information about the other forcefield files. Open the `topol.top` file and add the following lines:

```
#include "../toppar/martini_v3.0.0.itp"
#include "../toppar/martini_v3.0.0_ffbonded_v2_openbeta.itp"
#include "molecule_0.itp"
#include "../toppar/martini_v3.0.0_phospholipids_PC_v2_openbeta.itp"
#include "../toppar/martini_v3.0.0_solvents_v1.itp"
#include "../toppar/martini_v3.0.0_ions_v1.itp"
```

... and delete the following lines.

```
#include "martini.itp"
#include "molecule_0.itp"
```


---

## Adding membrane to our system

Next step is to embed the protein in the membrane. For this, we use a tool called `insane` (**IN**sert membr**ANE**) which will create a lipid bilayer of our choice (we will use POPC in this tutorial) and embed the protein. Before embedding the protein, make sure the center of the transmembrane domain of the protein is at (0,0,0) and the domain is properly aligned so that when it is inside the membrane it has the correct orientation with respect to the membrane.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
insane -f CG.pdb -p temp.top -o memb.gro -l POPC -u POPC -box 10,10,17  

| Option           | Description                                                                                                                                             |
| ---------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------- |
| `-f aligned.pdb` | Specifies the input structure (typically a coarse-grained protein) that has already been oriented correctly with respect to the membrane.               |
| `-p temp.top`    | Outputs a temporary topology file containing the lipid composition and system information. This file is usually merged with the protein topology later. |
| `-o memb.gro`    | Writes the complete system coordinates in GROMACS `.gro` format, including the protein and lipid bilayer.                                               |
| `-l POPC`        | Sets the lipid type for the **lower leaflet** of the membrane to POPC.                                                                                  |
| `-u POPC`        | Sets the lipid type for the **upper leaflet** of the membrane to POPC.                                                                                  |
| `-box 10,10,17`  | Defines the simulation box dimensions as **10 × 10 × 17 nm** in the x, y, and z directions, respectively.                                               |

---

We need to copy the information about the number of lipid molecules from the temporary topology file, `temp.top`, to the previous `topol.top` for the next steps. For example, the last two lines of `temp.top` show the lipid type in the upper and lower leaflet and the corresponding number of lipids. Copy these two lines into `topol.top`:

```
POPC           165
POPC           165
```

NOTE: Make sure that the order of the molecules in the `[molecules]` section is the same as in the structure file (`.gro` or `.pdb`).

## Adding water and ions

Now we have to add water and ions to the simulation box to make the system behave more like real biological or chemical environments. In MARTINI, one water bead is equivalent to four all-atom water molecules. To add water, we use the `solvate` command and a structure file of water provided as `water.gro`.

In [22]:
%%bash
# Fix topol.top after martinize2: replace placeholder includes and add lipids.
# IMPORTANT: set POPC counts to the values printed by insane in temp.top
# (last lines of temp.top: upper leaflet then lower leaflet).
N_UPPER=$(tail -2 temp.top | head -1 | awk '{print $2}')
N_LOWER=$(tail -1 temp.top | awk '{print $2}')
echo "POPC upper=$N_UPPER  lower=$N_LOWER"

cat > topol.top << EOF
#include "../toppar/martini_v3.0.0.itp"
#include "../toppar/martini_v3.0.0_ffbonded_v2_openbeta.itp"
#include "molecule_0.itp"
#include "../toppar/martini_v3.0.0_phospholipids_PC_v2_openbeta.itp"
#include "../toppar/martini_v3.0.0_solvents_v1.itp"
#include "../toppar/martini_v3.0.0_ions_v1.itp"

[ system ]
Title of the system in water

[ molecules ]
molecule_0    1
POPC        ${N_UPPER}
POPC        ${N_LOWER}
EOF

echo '--- topol.top now ---'
cat topol.top

POPC upper=167  lower=165
--- topol.top now ---
#include "../toppar/martini_v3.0.0.itp"
#include "../toppar/martini_v3.0.0_ffbonded_v2_openbeta.itp"
#include "molecule_0.itp"
#include "../toppar/martini_v3.0.0_phospholipids_PC_v2_openbeta.itp"
#include "../toppar/martini_v3.0.0_solvents_v1.itp"
#include "../toppar/martini_v3.0.0_ions_v1.itp"

[ system ]
Title of the system in water

[ molecules ]
molecule_0    1
POPC        167
POPC        165


In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
gmx solvate -cp memb.gro  -cs ../inp_files/water.gro -o solvated.gro -radius 0.21 -p topol.top

| Option            | Description                                                                                                                                                  |
| ----------------- | ------------------------------------------------------------------------------------------------------------------------------------------------------------ |
| `-cp memb.gro`    | Input coordinate file of the system to be solvated. Here, it contains the membrane + protein system.                                                        |
| `-cs water.gro`   | Defines the solvent configuration file. In MARTINI workflows, this is often coarse-grained water (e.g., polarizable or standard MARTINI water).           |
| `-o solvated.gro` | Output coordinate file after adding water molecules. This is the fully solvated system.                                                                     |
| `-radius 0.21`    | Sets the minimum distance (in nm) between solvent molecules and any solute bead. Water molecules closer than this cutoff are removed to avoid overlaps.    |
| `-p topol.top`    | Updates the topology file by adding the correct number of solvent molecules.                                                                                |

---

This will generate a new coordinate file (`solvated.gro`) with added water and updated topology file, `topol.top`, with the number of water beads added.

**Next we will add ions.** This is to neutralise the total charge of the system and reproduce a physiological ion environment. This is a two-step process. The first step creates a binary input file (`.tpr`) needed for ion replacement. The second step uses `genion` to replace water beads with sodium (`NA`) and chloride (`CL`) ions at a given concentration (`-conc 0.15`). This will generate a coordinate file with ions (`ionized.gro`) and an updated `topol.top` with the number of ions added.


In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
gmx grompp -f ../inp_files/CG_minim.mdp -c solvated.gro -p topol.top -o ionize
echo W | gmx genion -s ionize.tpr -o ionized -p topol.top -pname NA -nname CL -neutral -conc 0.15

`echo W | gmx ...` is used to automatically pass the option `W` to the prompt, which will replace the molecule "W" with "ION". Without `echo W |`, GROMACS would prompt you interactively.


# Molecular Dynamics Simulation

After the system is prepared, we can now proceed to the simulation. In this part, we need to provide an important file (`.mdp`), which contains parameter settings for each step. This file determines the conditions and type of simulation we want to run.


## Minimisation

The first step is to ensure that the energy of our system is minimized, or close to the minimum energy state. When we build the system, there is a chance that some atoms are placed too close together, which can cause steric clashes and unrealistic high energies.

The minimisation step moves atoms **downhill on the potential energy surface** to find a nearby low-energy, stable configuration. This step is used to remove *bad atomic contacts* and put the system into a physically reasonable starting shape before running dynamics.

The first step, `grompp`, generates a binary input file (`.tpr`) required for running energy minimisation. The second step, `mdrun`, performs the actual energy minimisation simulation.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
gmx grompp -f ../inp_files/CG_minim.mdp -c ionized.gro -p topol.top -o emsol.tpr
gmx mdrun -deffnm emsol #-v

| Option (grompp)  | Meaning                                               |
| ---------------- | ----------------------------------------------------- |
| `-f minim.mdp`   | Energy minimization parameters.                       |
| `-c ionized.gro` | Starting structure (protein + lipids + water + ions). |
| `-p topol.top`   | System topology.                                      |
| `-o emsol.tpr`   | Output run input file for `mdrun`.                    |

| Option (mdrun)  | Meaning                                                |
| --------------- | ------------------------------------------------------ |
| `-deffnm emsol` | Sets the default filename prefix for all output files. |
| `-v`            | Verbose mode (prints progress of minimization).        |

Using `-deffnm emsol`, GROMACS generates:

- `emsol.gro` : final minimized structure  
- `emsol.log` : log of the minimisation process  
- `emsol.edr` : energy file (useful for analysis)  
- `emsol.trr` : trajectory of minimisation steps  
- `emsol.tpr` : copy of input run file  

---

## Equilibration

The next step of the simulation is equilibration. This is to bring the system into a **stable, physically realistic state** before collecting meaningful data in the production run. In this process, we bring the system to the target temperature and pressure in a step-by-step manner and prepare it for the final production run.

### Create index file

Before that, we need to create an *index file*, which is used to define and group specific sets of atoms so we can easily select and work with parts of our system during analysis or simulations.

In the equilibration steps, we need to define groups for separate thermostat and barostat. In our system, we created four index groups: all atoms, protein, membrane, and solvent (water + ions). Proteins, membranes, and solvent are assigned separate thermostats.

Usually, a GROMACS tool `gmx make_ndx` is used to create the index file, which is interactive and must be done manually each time. However, to automate the process, we use the `MDAnalysis` tool to create the index file (`index.ndx`).

In [ ]:
%%bash
# Build index.ndx natively with GROMACS (MDAnalysis not available in the workshop venv).
# NOTE: group numbers below (1=Protein, 13=POPC, 14=W, 15=ION) come from the
# make_ndx listing for THIS system. If you change protein/membrane, re-run
# 'gmx make_ndx -f emsol.tpr' with only 'q' first to read the correct numbers.
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
gmx make_ndx -f emsol.tpr -o index.ndx << 'EOF'
14 | 15
name 16 SOL
name 1 PROT
name 13 MEM
q
EOF

## NVT Equilibration

**NVT equilibration** is used in molecular dynamics to stabilize the system at a **fixed number of particles (N), volume (V), and temperature (T)** before moving to pressure equilibration (NPT) and production runs.

After minimisation, the system is still not physically stable. NVT equilibration focuses on bringing the temperature to the target value in a controlled way.

This is done using a thermostat, which slowly adjusts the kinetic energy of the system to reach the desired temperature.

Like every simulation step, it has two stages. First, we generate a `.tpr` file for the next `mdrun` step.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
# Uses the short CG_nvt_tut.mdp from the setup cell (0.5 ns demo run).
gmx grompp -f CG_nvt_tut.mdp -c emsol.gro -r emsol.gro -p topol.top -n index.ndx -o nvt.tpr
gmx mdrun -deffnm nvt -nb gpu -bonded gpu

Note that in the `grompp` step, we used an extra parameter apart from `-n index.ndx` — `-r emsol.gro`. This is important for **position restraints** for the reference structure. It helps the structure not move too much during the equilibration steps compared to the solvent.

---

## NPT Equilibration

**NPT equilibration** is used in molecular dynamics to bring the system to the correct **pressure and density**, after temperature has already been stabilized in NVT.

In NPT, the system is kept at constant number of particles (N), pressure (P), and temperature (T).

This step adjusts the system density, distributes water molecules more evenly, and allows the box size to change to remove vacuum gaps or overly compressed regions.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
# Uses the short CG_npt_tut.mdp from the setup cell (1 ns demo run).
# -t nvt.cpt carries velocities over from NVT for a smooth continuation.
gmx grompp -f CG_npt_tut.mdp -c nvt.gro -r nvt.gro -p topol.top -t nvt.cpt -n index.ndx -o npt.tpr -maxwarn 1
gmx mdrun -deffnm npt -nb gpu -bonded gpu

Here, we added another parameter, `-t nvt.cpt`, which tells the program to use the velocities of the particles from the last simulation. This allows the simulation to continue smoothly from NVT and prevents the generation of random velocities for the particles.

---

## Production run

This is the final stage of the simulation workflow. After energy minimization and equilibration (NVT + NPT), the system is now ready for meaningful data collection. The system is now allowed to explore real dynamics, and no position restraints are used on the atoms.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
# Uses the short CG_prod_tut.mdp from the setup cell (5 ns demo run).
gmx grompp -f CG_prod_tut.mdp -c npt.gro -p topol.top -n index.ndx -o prod.tpr -maxwarn 1
gmx mdrun -deffnm prod -nb gpu -bonded gpu

# Post Processing

Before post processing, we have to create another index file. This helps in analysing different domains of the proteins, membranes, etc.

To make it easier to work with and to speed up the analysis, it is better to remove solvent (when we are not dealing with analysis related to water/ions). Solvent takes up a lot of space and can reduce the speed of post-processing.

For better organisation of files, we also create another folder, e.g., `analysis`.

In [12]:
# Optional: a folder to keep analysis outputs tidy.
# NOTE: we stay in the exercise/ directory for all commands below,
# because 'cd' in a Python cell does NOT carry over to %%bash cells.
import os
os.makedirs('analysis', exist_ok=True)
print('analysis/ ready; staying in', os.getcwd())

analysis/ ready; staying in /net/tscratch/people/tutorial113/workspaces/2026-06-30-mcb-eebg/notebooks/EEBG_workshop/CG_Membrane/exercise


## Creating index file

In this `index_analysis.ndx` file, we have different groups of atoms such as protein, membranes, transmembrane domain of the protein, etc. This helps to analyse the behaviour of different groups with each other.

In [ ]:
%%bash
# Build index_analysis.ndx natively (MDAnalysis/pkg_resources unavailable in the venv).
# Group numbers from 'gmx make_ndx -f prod.tpr': 1=Protein, 13=POPC, 14=W, 15=ION.
# Transmembrane = atoms 242-312 (equals MDAnalysis 'bynum 242:312').
# not_SOL = everything except water and ions (protein + membrane).
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
gmx make_ndx -f prod.tpr -o index_analysis.ndx << 'EOF'
name 1 PROT
a 242-312
name 16 Transmembrane
name 13 MEM
! 14 & ! 15
name 17 not_SOL
q
EOF

## Correction of PBC

During MD simulation, periodic boundary conditions are used such that molecules leaving one side of the box re-enter from the opposite side. This creates an effectively infinite system, which is desirable.

However, for certain analysis or visualisation, this can result in “broken” molecules or fragmented trajectories.

To fix this, we apply a correction to make the molecules whole again. We use a GROMACS tool called `trjconv` for this.

In [ ]:
%%bash
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
# Centre on the transmembrane domain, make molecules whole, strip solvent.
# Redirect feeds two selections: 'Transmembrane' (centering), 'not_SOL' (output).
gmx trjconv -f prod.xtc -s prod.tpr -o prod_noW.xtc -n index_analysis.ndx -center -pbc mol -ur compact <<< "Transmembrane not_SOL"

| File                    | Meaning                                                |
| ----------------------- | ------------------------------------------------------ |
| `-f ../prod.xtc`        | Raw production trajectory                              |
| `-s ../prod.tpr`        | Run input file (provides structure/topology reference) |
| `-o prod_noW.xtc`       | Output cleaned trajectory                              |
| `-n index_analysis.ndx` | Index file defining custom groups                      |
| `-center`              | Centers on a chosen group (here **Transmembrane**)     |
| `-pbc mol`             | Keeps whole molecules intact                           |
| `-ur compact`          | Minimizes box volume representation                    |

`<<< "Transmembrane not_SOL"`: This is bash input redirection, not a GROMACS option. It automatically provides answers to the interactive prompts from `trjconv` when it asks for group selections — here, **"Transmembrane"** for centering and **"not_SOL"** (all atoms except solvent) for output.

In [ ]:
%%bash
# Make a solvent-free single-frame structure for visualization.
module load GCC/14.2.0 GROMACS/2025.4-CUDA-12.8.0-SMP
# take the first frame of the cleaned trajectory as a .gro (or .pdb) snapshot
echo "not_SOL" | gmx trjconv -f prod_noW.xtc -s prod.tpr -n index_analysis.ndx -o prod_vis.pdb -dump 0
ls -la prod_vis.pdb

In [17]:
# Visualization with nglview.
# Two issues are worked around here, both caused by the shared read-only venv,
# not by the simulation:
#   1) pkg_resources is missing (setuptools absent in this Python 3.13 venv),
#      so nglview fails to import. We inject a tiny stub before importing it.
#   2) the inline JupyterLab widget version is mismatched (backend wants
#      nglview-js-widgets 3.1.4, the Hub has 3.1.5), so the inline viewer
#      will not render. We export a standalone HTML file instead and open it
#      in the browser (right-click view.html in the file browser -> Download).
import sys, types

# --- stub pkg_resources so nglview can import ---
if 'pkg_resources' not in sys.modules:
    stub = types.ModuleType('pkg_resources')
    class _Dist:
        version = '0'
    stub.get_distribution = lambda *a, **k: _Dist()
    stub.DistributionNotFound = Exception
    stub.resource_filename = lambda *a, **k: ''
    sys.modules['pkg_resources'] = stub

import nglview as nv

# Visualize the actual pipeline output (solvent stripped), not the nonexistent aa2.pdb.
# Martini is coarse-grained, so 'cartoon' does not apply; we render beads instead.
view = nv.show_file('prod_vis.pdb')
view.clear_representations()
view.add_representation('point', selection='all')
view.add_representation('spacefill', selection='not (resname POPC)', radius=0.6)

# Export to a standalone HTML file (bypasses the JupyterLab widget version mismatch).
nv.write_html('view.html', [view])
print('Wrote view.html - right-click it in the file browser and Download, then open locally.')

Wrote view.html - right-click it in the file browser and Download, then open locally.
